In [1]:
import requests
import json

In [2]:
with open("../secrets.json") as f:
    secrets = json.load(f)
api_key = secrets['jsearch_api_key']

In [3]:
import requests

class JobReport:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://jsearch.p.rapidapi.com/search"
        
    def JobSearch(self, job_title, location=None, years_of_experience=None, field=None, date_posted=None):
        """
        Search for jobs using the JSearch API
        
        Parameters:
        - job_title: The job title to search for
        - location: Optional city, state, or country
        - years_of_experience: Optional experience level (e.g., "entry_level", "mid_level", "senior")
        - field: Optional field/industry (e.g., "technology", "finance")
        - date_posted: Optional time frame for when job was posted (e.g., "all", "today", "3days", "week", "month")
        
        Returns:
        - Dictionary containing job listings data
        """
        headers = {
            "X-RapidAPI-Key": self.api_key,
            "X-RapidAPI-Host": "jsearch.p.rapidapi.com"
        }
        
        # Build query string based on parameters
        query = job_title
        if field:
            query += f" {field}"
        
        params = {
            "query": query,
            "page": 1,
            "num_pages": 1
        }
        
        if location:
            params["location"] = location
        
        # Date posted filter
        if date_posted:
            # JSearch supports values like "all", "today", "3days", "week", "month"
            params["date_posted"] = date_posted
        
        # Experience level mapping
        if years_of_experience:
            if years_of_experience in ["entry_level", "mid_level", "senior"]:
                exp_terms = {
                    "entry_level": "entry level junior",
                    "mid_level": "mid level",
                    "senior": "senior experienced"
                }
                query += f" {exp_terms[years_of_experience]}"
            else:
                query += f" {years_of_experience} years experience"
            
            params["query"] = query
        
        try:
            response = requests.get(self.base_url, headers=headers, params=params)
            response.raise_for_status()
            data = response.json()
            
            if "data" not in data:
                return {"jobs": [], "count": 0, "message": "No jobs found"}
            
            # Process and format job listings
            job_listings = []
            for job in data["data"]:
                job_data = {
                    "job_id": job.get("job_id", ""),
                    "title": job.get("job_title", ""),
                    "company": job.get("employer_name", ""),
                    "location": job.get("job_city", "") + ", " + job.get("job_country", ""),
                    "description": job.get("job_description", "")[:300] + "..." if job.get("job_description") else "",
                    "salary": job.get("job_salary", job.get("job_min_salary", "")),
                    "url": job.get("job_apply_link", ""),
                    "posted_at": job.get("job_posted_at_datetime_utc", ""),
                    "job_type": job.get("job_employment_type", "")
                }
                job_listings.append(job_data)
            
            return {
                "jobs": job_listings,
                "count": len(job_listings),
                "message": f"Found {len(job_listings)} jobs matching '{query}'"
            }
            
        except requests.exceptions.RequestException as e:
            print(f"Error fetching job data: {str(e)}")
            return {"jobs": [], "count": 0, "message": f"Error: {str(e)}"}
    
    def pretty_print_jobs(self, jobs_data):
        """
        Format and print job listings in a readable format
        
        Parameters:
        - jobs_data: Dictionary returned by the JobSearch method
        """
        if jobs_data["count"] == 0:
            print(jobs_data["message"])
            return
        
        print(f"\n📋 JOB SEARCH RESULTS: {jobs_data['message']}")
        print("=" * 80)
        
        for i, job in enumerate(jobs_data["jobs"], 1):
            print(f"\n{i}. {job['title']}")
            print("-" * 80)
            print(f"🏢 Company: {job['company']}")
            print(f"📍 Location: {job['location']}")
            
            if job['job_type']:
                print(f"💼 Type: {job['job_type']}")
                
            if job['salary']:
                print(f"💰 Salary: {job['salary']}")
                
            if job['posted_at']:
                print(f"📅 Posted: {job['posted_at']}")
                
            print(f"\n📝 Description: {job['description']}")
            print(f"\n🔗 Apply: {job['url']}")
            print("-" * 80)
        
        print(f"\nFound {jobs_data['count']} jobs matching your criteria.")

In [5]:
job = "Data Scientist"
location = "seattle"
field = "machine learning"

jobsearch = JobReport(api_key)

search_results = jobsearch.JobSearch(
    job_title=job, 
    location=location,
    field=field
)

jobsearch.pretty_print_jobs(search_results)


📋 JOB SEARCH RESULTS: Found 10 jobs matching 'Data Scientist machine learning'

1. Senior Data Scientist - Customer Data Machine Learning Team
--------------------------------------------------------------------------------
🏢 Company: Capital One
📍 Location: McLean, US
💼 Type: Full-time and Part-time
📅 Posted: 2025-03-10T00:00:00.000Z

📝 Description: Senior Data Scientist - Customer Data Machine Learning Team

Data is at the center of everything we do. As a startup, we disrupted the credit card industry by individually personalizing every credit card offer using statistical modeling and the relational database, cutting edge technology in 1988! F...

🔗 Apply: https://www.capitalonecareers.com/job/mclean/senior-data-scientist-customer-data-machine-learning-team/1732/78579920880?utm_campaign=google_jobs_apply&utm_source=google_jobs_apply&utm_medium=organic
--------------------------------------------------------------------------------

2. Junior Data Scientist
--------------------------

In [6]:
search_results

{'jobs': [{'job_id': 'ezQBhmuTbw9umR1hAAAAAA==',
   'title': 'Senior Data Scientist - Customer Data Machine Learning Team',
   'company': 'Capital One',
   'location': 'McLean, US',
   'description': 'Senior Data Scientist - Customer Data Machine Learning Team\n\nData is at the center of everything we do. As a startup, we disrupted the credit card industry by individually personalizing every credit card offer using statistical modeling and the relational database, cutting edge technology in 1988! F...',
   'salary': None,
   'url': 'https://www.capitalonecareers.com/job/mclean/senior-data-scientist-customer-data-machine-learning-team/1732/78579920880?utm_campaign=google_jobs_apply&utm_source=google_jobs_apply&utm_medium=organic',
   'posted_at': '2025-03-10T00:00:00.000Z',
   'job_type': 'Full-time and Part-time'},
  {'job_id': '0z8-fFqL4jNuUYLXAAAAAA==',
   'title': 'Junior Data Scientist',
   'company': 'Battelle',
   'location': 'Arlington, US',
   'description': 'Battelle delivers 